Implements per-pair decoding

Supports:
    Pairwise flows; Multi-frame input; Temporal conditioning

No operator implementation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=not norm)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.1, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [ ]:
class ContextNet(nn.Module):
    def __init__(self, in_ch, hidden_ch=128):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=2),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=4),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=8),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=16),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=1),
            nn.Conv2d(hidden_ch, 2, kernel_size=3, padding=1),
        )
    def forward(self, x):
        return self.net(x)
    

In [ ]:
class FlowDecoder_single(nn.Module):
    """
    Input:
        embeddings z: [B, C, H, W]
    
    Output:
        flow: [B, 2, H_out, W_out]
    """
    def __init__(self, in_ch=128, hidden_ch=128, upsample=4, use_prev_flow=True):
        super().__init__()
        self.upsample = upsample
        self.use_prev_flow = use_prev_flow
        
        self.conv0 = nn.Sequential(
            ConvBlock(in_ch, hidden_ch),
            ConvBlock(hidden_ch, hidden_ch),
        )
        
        self.flow_head = nn.Conv2d(hidden_ch, 2, kernel_size=3, padding=1)
        self.refine = nn.Sequential(
            ConvBlock(hidden_ch + 2, hidden_ch),
            ConvBlock(hidden_ch, hidden_ch),
        )
        self.delta_head = nn.Conv2d(hidden_ch, 2, kernel_size=3, padding=1)
        self.context_net = ContextNet(hidden_ch)
        
    def forward(self, z, flow_prev=None):
        B, C, H, W = z.shape
        
        if self.use_prev_flow:
            if flow_prev is None:
                flow_prev = torch.zeros(B, 2, H * self.upsample, W * self.upsample).to(z.device)
                flow_prev_downSample = F.interpolate(flow_prev, size=(H, W), mode='bilinear', align_corners=False)
                z = torch.cat((z, flow_prev_downSample), dim=1)
        
        feat = self.conv0(z)
        flow = self.flow_head(feat)
        
        flow_up = F.interpolate(flow, scale_factor=self.upsample, mode='bilinear', align_corners=False)
        feat_up = F.interpolate(feat, scale_factor=self.upsample, mode='bilinear', align_corners=False)
        
        x = torch.cat([feat_up, flow_up], dim=1)
        x = self.refine(x)
        
        delta_flow = self.delta_head(x)
        flow_refined = flow_up + delta_flow
        
        context_delta = self.context_net(x)
        flow_final = flow_refined + context_delta
        
        return flow_final

In [ ]:
class FlowDecoder(nn.Module):
    """
    Input:
        fused_seq: [B, Tm, C, H, W]
    Output:
        flows: [B, Tm, 2, H_out, W_out]
    """
    
    def __init__(self, in_ch=128, hidden_ch=128, upsample=4, use_prev_flow=True):
        super().__init__()
        self.decoder = FlowDecoder_single(in_ch, hidden_ch, upsample, use_prev_flow)
        self.use_prev_flow = use_prev_flow
    
    def forward(self, fused_seq):
        B, Tm, C, H, W = fused_seq.shape
        
        flows = []
        flow_prev = None
        for t in range(Tm):
            z = fused_seq[:, t]
            if self.use_prev_flow:
                flow_t = self.decoder(z, flow_prev)
            else:
                flow_t = self.decoder(z)
            flows.append(flow_t)
            flow_prev = flow_t.detach()
        flows = torch.stack(flows, dim=1)
        
        return flows